# Stage 6B — Controlled Official-Source Acquisition

This notebook converts the Stage 6A acquisition universe into a controlled document-level reference registry for strategy and company-result evidence.

Stage 6B acquires citable references and provenance only. It does not republish copyrighted source documents, extract analytical observations, infer strategy effectiveness, create strategy-result links, rank companies, build a composite score, select an overall winner, or prepare a report/README.


## Environment Setup

Import the required libraries, lock the authoritative Stage 6A commit and canonical input checksums, and define deterministic runtime paths and the retrieval date.


In [1]:
from __future__ import annotations

import hashlib
import os
import shutil
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path

import pandas as pd

REPOSITORY_FULL_NAME = "Ronaldo-spec/indonesia-fmcg-brand-portfolio-analysis"
INPUT_COMMIT = "51e5b3d9459469aa34c91adcc2fd2d4573a0170c"
RETRIEVAL_DATE = "2026-08-21"

INPUT_LOCKS = {
    "metadata/stage6a_input_lock.csv":
        "8fbe53b4f36860f8edf52883c4f02c77e23bb0c9f27e79c8d22ead8b008f8877",
    "metadata/stage6a_acquisition_universe.csv":
        "8392a86c1a67106e8301a253c60f66c6afd6eca405cdaed87960482988da61eb",
    "metadata/stage6a_source_inventory.csv":
        "6b3dc1b25fa13c0c183825e9bbdc5df4e4555540ac85144aea6a79ffabbd3928",
    "metadata/stage6a_access_redistribution_audit.csv":
        "e38ebcd9f1127d4a762007dff3a0be7ea6fb5b0568beacfc8694c04615362efa",
    "metadata/stage6a_acquisition_validation.csv":
        "6eb656bc24c8b9d8dbbb0bbaf2979e429f01e5057159c8e5cc289a18dc75ffe1",
}

OUTPUT_ROOT = Path(
    os.environ.get("FMCG_STAGE6B_OUTPUT_ROOT", "/content/fmcg_stage6b_outputs")
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

print(f"Locked Stage 6A commit: {INPUT_COMMIT}")
print(f"Required Stage 6A inputs: {len(INPUT_LOCKS)}")
print(f"Output root: {OUTPUT_ROOT}")


Locked Stage 6A commit: 51e5b3d9459469aa34c91adcc2fd2d4573a0170c
Required Stage 6A inputs: 5
Output root: /content/fmcg_stage6b_outputs


## Locked Stage 6A Input Retrieval

Retrieve only the five canonical Stage 6A metadata artifacts from the authoritative Stage 6A commit. Colab uses the `GITHUB_TOKEN` secret; a local validation root can be supplied through `FMCG_STAGE6B_INPUT_ROOT`.


In [2]:
configured_root = os.environ.get("FMCG_STAGE6B_INPUT_ROOT")

if configured_root:
    INPUT_ROOT = Path(configured_root)
    input_mode = "local_validation_root"
else:
    try:
        from google.colab import userdata
    except ImportError as exc:
        raise RuntimeError(
            "Run in Google Colab or set FMCG_STAGE6B_INPUT_ROOT for local validation."
        ) from exc

    github_token = userdata.get("GITHUB_TOKEN")
    if not github_token:
        raise RuntimeError(
            "Colab Secret GITHUB_TOKEN is unavailable or access has not been granted."
        )

    INPUT_ROOT = Path("/content/fmcg_stage6b_inputs")
    if INPUT_ROOT.exists():
        shutil.rmtree(INPUT_ROOT)
    INPUT_ROOT.mkdir(parents=True, exist_ok=True)

    for relative_path in INPUT_LOCKS:
        encoded_path = urllib.parse.quote(relative_path, safe="/")
        url = (
            f"https://api.github.com/repos/{REPOSITORY_FULL_NAME}"
            f"/contents/{encoded_path}?ref={INPUT_COMMIT}"
        )
        request = urllib.request.Request(
            url,
            headers={
                "Authorization": f"Bearer {github_token}",
                "Accept": "application/vnd.github.raw+json",
                "X-GitHub-Api-Version": "2022-11-28",
                "User-Agent": "fmcg-stage6b-colab",
            },
        )

        destination = INPUT_ROOT / relative_path
        destination.parent.mkdir(parents=True, exist_ok=True)

        try:
            with urllib.request.urlopen(request, timeout=60) as response:
                destination.write_bytes(response.read())
        except urllib.error.HTTPError as exc:
            raise RuntimeError(
                f"GitHub input retrieval failed for {relative_path} "
                f"with HTTP {exc.code}."
            ) from exc

    del github_token
    input_mode = "locked_github_commit"

missing = [
    relative_path
    for relative_path in INPUT_LOCKS
    if not (INPUT_ROOT / relative_path).exists()
]

if missing:
    raise FileNotFoundError(f"Missing required Stage 6A inputs: {missing}")

print(f"Input mode: {input_mode}")
print(f"Required files found: {len(INPUT_LOCKS)}/{len(INPUT_LOCKS)}")


Input mode: locked_github_commit
Required files found: 5/5


## Input Integrity and Prior-Stage Gate

Verify every inherited Stage 6A artifact against its locked SHA-256 value and confirm that the Stage 6A gate remains `PASS_WITH_CAVEAT` before constructing the document registry.


In [3]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file_handle:
        for chunk in iter(lambda: file_handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


lock_rows = []

for relative_path, expected_sha256 in INPUT_LOCKS.items():
    actual_sha256 = sha256_file(INPUT_ROOT / relative_path)
    lock_rows.append(
        {
            "file_path": relative_path,
            "expected_sha256": expected_sha256,
            "actual_sha256": actual_sha256,
            "hash_match": actual_sha256 == expected_sha256,
            "locked_repository_commit": INPUT_COMMIT,
        }
    )

stage6b_input_lock = pd.DataFrame(lock_rows)

if not stage6b_input_lock["hash_match"].all():
    raise RuntimeError("One or more Stage 6A input checksums failed.")

stage6a_universe = pd.read_csv(
    INPUT_ROOT / "metadata/stage6a_acquisition_universe.csv",
    dtype=str,
    keep_default_na=False,
)
stage6a_sources = pd.read_csv(
    INPUT_ROOT / "metadata/stage6a_source_inventory.csv",
    dtype=str,
    keep_default_na=False,
)
stage6a_legal = pd.read_csv(
    INPUT_ROOT / "metadata/stage6a_access_redistribution_audit.csv",
    dtype=str,
    keep_default_na=False,
)
stage6a_validation = pd.read_csv(
    INPUT_ROOT / "metadata/stage6a_acquisition_validation.csv",
    dtype=str,
    keep_default_na=False,
)

final_stage6a = stage6a_validation.loc[
    stage6a_validation["check_id"] == "S6A027"
].iloc[0]

if (
    final_stage6a["result"] != "PASS_WITH_CAVEAT"
    or final_stage6a["status"] != "passed_with_caveat"
):
    raise RuntimeError(
        "Stage 6A final gate is not the expected PASS_WITH_CAVEAT status."
    )

if len(stage6a_universe) != 26:
    raise RuntimeError(
        f"Expected 26 Stage 6A acquisition targets, found {len(stage6a_universe)}."
    )

print(
    f"Input checksums passed: "
    f"{stage6b_input_lock['hash_match'].sum()}/{len(stage6b_input_lock)}"
)
print(
    f"Stage 6A gate: "
    f"{final_stage6a['result']} / {final_stage6a['status']}"
)
print(f"Inherited acquisition targets: {len(stage6a_universe)}")


Input checksums passed: 5/5
Stage 6A gate: PASS_WITH_CAVEAT / passed_with_caveat
Inherited acquisition targets: 26


## Controlled Document Registry

Register only verified official pages, official document references, official index entries, and narrowly permitted contextual references. An index entry is not silently upgraded to a resolved direct file. All repository storage remains reference-only.


In [4]:
registry_columns = [
    "document_id",
    "canonical_group",
    "reporting_entity",
    "stage6a_target_ids",
    "source_family",
    "document_title",
    "source_type",
    "reference_period",
    "publication_date",
    "geography",
    "source_url",
    "direct_file_url",
    "resolution_status",
    "acquisition_status",
    "priority_tier",
    "evidence_role",
    "claim_label",
    "ownership_scope",
    "comparability_treatment",
    "repository_storage_policy",
    "legal_treatment",
    "retrieval_date",
    "notes",
]

registry_rows = [
    # ------------------------------------------------------------------
    # Wings Group
    # ------------------------------------------------------------------
    (
        "S6BDOC_WNG_001", "Wings Group", "Wings Group",
        "S6A_T02;S6A_T05", "current_portfolio_context", "About Us",
        "official_corporate_webpage", "current_page", "", "Indonesia",
        "https://wingscorp.com/about-us/", "",
        "official_page_verified", "verified_reference_only", "2",
        "current operating and ownership context", "company_reported",
        "group-level context; named JVs require separate treatment",
        "current page only; do not backdate",
        "reference_only", "reference_only_pending_terms",
        RETRIEVAL_DATE,
        "Current corporate context only; not evidence of historical portfolio continuity.",
    ),
    (
        "S6BDOC_WNG_002", "Wings Group", "Wings Group",
        "S6A_T02", "current_portfolio_context", "Local Brands",
        "official_corporate_webpage", "current_page", "", "Indonesia",
        "https://wingscorp.com/local-brands/", "",
        "official_page_verified", "verified_reference_only", "2",
        "current consumer-brand portfolio context", "company_reported",
        "strict-control status must follow ownership registry rather than webpage grouping alone",
        "current page only; do not backdate",
        "reference_only", "reference_only_pending_terms",
        RETRIEVAL_DATE,
        "Used only as current portfolio context.",
    ),
    (
        "S6BDOC_WNG_003", "Wings Group", "Wings Group",
        "S6A_T01", "strategy_events", "News & Activities Archive",
        "official_corporate_news_index", "2022–2025 archive", "", "Indonesia",
        "https://wingscorp.com/berita-kegiatan/", "",
        "official_page_verified", "verified_reference_only", "2",
        "dated strategy-event discovery", "company_reported",
        "group or named operating unit as stated in each article",
        "archive coverage is not assumed complete",
        "reference_only", "reference_only_pending_terms",
        RETRIEVAL_DATE,
        "Article-level evidence is required for any later extraction.",
    ),
    (
        "S6BDOC_WNG_004", "Wings Group", "Wings Care",
        "S6A_T01", "strategy_events", "Wings Care launches ProGuard",
        "official_corporate_press_release", "2022_action", "2022-08-18", "Indonesia",
        "https://wingscorp.com/wings-care-luncurkan-proguard-the-next-level-of-antibacterial-body-wash-guna-lawan-mutasi-kuman-virus-dan-bakteri/",
        "",
        "direct_page_verified", "acquired_reference", "2",
        "product launch and campaign action", "company_reported",
        "Wings Care action; no separate legal-entity attribution inferred",
        "action evidence only; no effectiveness inference",
        "reference_only", "reference_only_pending_terms",
        RETRIEVAL_DATE,
        "Eligible later as dated product/marketing action evidence only.",
    ),
    (
        "S6BDOC_WNG_005", "Wings Group", "WINGS Food",
        "S6A_T01", "strategy_events", "GOLDA Cappuccino launch",
        "official_corporate_press_release", "2022_action", "2022-04-08", "Indonesia",
        "https://wingscorp.com/lengkapi-koleksinya-di-pasar-kopi-rtd-wings-food-luncurkan-golda-cappuccino/",
        "",
        "direct_page_verified", "acquired_reference", "2",
        "product, pack-price and channel action", "company_reported",
        "WINGS Food action; no separate legal-entity attribution inferred",
        "source-embedded market-position claims remain company-reported",
        "reference_only", "reference_only_pending_terms",
        RETRIEVAL_DATE,
        "The article provides dated launch, price/pack and channel information.",
    ),
    (
        "S6BDOC_WNG_006", "Wings Group", "WINGS Food",
        "S6A_T01", "strategy_events", "Ale-Ale FunFlava Cocopandan launch",
        "official_corporate_press_release", "2023_action", "2023-03-23", "Indonesia",
        "https://wingscorp.com/ramadhan-tiba-wings-food-luncurkan-ale-ale-funflava-cocopandan/",
        "",
        "direct_page_verified", "acquired_reference", "2",
        "product launch and target-consumer action", "company_reported",
        "WINGS Food action; no separate legal-entity attribution inferred",
        "action evidence only; no effectiveness inference",
        "reference_only", "reference_only_pending_terms",
        RETRIEVAL_DATE,
        "Any leadership language inside the release remains company-reported.",
    ),
    (
        "S6BDOC_WNG_007", "Wings Group", "WINGS Food",
        "S6A_T01", "strategy_events", "ISOPLUS COCO launch",
        "official_corporate_press_release", "2023_action", "2023-04-02", "Indonesia",
        "https://wingscorp.com/wings-food-luncurkan-isoplus-coco-excellent-hydration-dengan-kesegaran-air-kelapa-muda-thailand/",
        "",
        "direct_page_verified", "acquired_reference", "2",
        "product launch and positioning action", "company_reported",
        "WINGS Food action; no separate legal-entity attribution inferred",
        "action evidence only; no effectiveness inference",
        "reference_only", "reference_only_pending_terms",
        RETRIEVAL_DATE,
        "Dated official product action.",
    ),
    (
        "S6BDOC_WNG_008", "Wings Group", "Wings Group",
        "S6A_T02", "current_portfolio_context", "Bumbu Kaldu Sedaap",
        "official_brand_webpage", "current_page", "2025-03-27", "Indonesia",
        "https://wingscorp.com/brand-detail/bumbu-kaldu-sedaap/", "",
        "current_page_verified", "verified_reference_only", "2",
        "current product context", "company_reported",
        "current brand-page context only",
        "page date is not treated as launch date",
        "reference_only", "reference_only_pending_terms",
        RETRIEVAL_DATE,
        "Do not use the current brand page as historical launch evidence.",
    ),
    (
        "S6BDOC_WNG_009", "Wings Group", "Wings Group",
        "S6A_T02", "current_portfolio_context", "AQUVIVA",
        "official_brand_webpage", "current_page", "2025-04-25", "Indonesia",
        "https://wingscorp.com/brand-detail/aquviva/", "",
        "current_page_verified", "verified_reference_only", "2",
        "current product context", "company_reported",
        "current brand-page context only",
        "page date is not treated as launch date",
        "reference_only", "reference_only_pending_terms",
        RETRIEVAL_DATE,
        "Current product context only.",
    ),

    # ------------------------------------------------------------------
    # Indofood parent
    # ------------------------------------------------------------------
    *[
        (
            f"S6BDOC_IDF_AR_{year}", "Indofood",
            "PT Indofood Sukses Makmur Tbk",
            "S6A_T06;S6A_T12;S6A_T13", "annual_reporting",
            f"Annual Report {year}", "annual_report",
            f"FY{year}", "", "consolidated / segment disclosures",
            "https://www.indofood.com/investor-relation/annual-report", "",
            "official_index_entry_verified", "verified_reference_only", "1",
            "annual strategy, entity-scope and company-result source",
            "company_reported",
            "parent consolidated with segment disclosures",
            "preserve parent/segment and consumer/upstream boundaries",
            "reference_only", "reference_only_no_repo_copy",
            RETRIEVAL_DATE,
            "Official annual-report index confirms the target-year report; direct file URL is intentionally unresolved in Stage 6B.",
        )
        for year in (2022, 2023, 2024, 2025)
    ],
    *[
        (
            f"S6BDOC_IDF_FS_{year}", "Indofood",
            "PT Indofood Sukses Makmur Tbk",
            "S6A_T07", "audited_financials",
            f"Annual Financial Statements {year}", "audited_financial_statement",
            f"FY{year}", "", "consolidated",
            "https://www.indofood.com/menu/financial-statements", "",
            "official_index_entry_verified", "verified_reference_only", "1",
            "audited result source", "company_reported_or_audited",
            "parent consolidated",
            "do not attribute consolidated results to individual brands",
            "reference_only", "reference_only_no_repo_copy",
            RETRIEVAL_DATE,
            "Official financial-statement index confirms the target year; direct file URL remains unresolved.",
        )
        for year in (2022, 2023, 2024, 2025)
    ],
    (
        "S6BDOC_IDF_009", "Indofood", "PT Indofood Sukses Makmur Tbk",
        "S6A_T08", "period_results", "Full-year financial results FY2022",
        "official_earnings_release", "FY2022", "2023-03-27", "consolidated",
        "https://www.indofood.com/menu/financial-press-releases/indofoods-full-year-financial-results-for-the-year-ended-31-december-2022",
        "",
        "direct_page_verified", "acquired_reference", "2",
        "period-result corroboration", "company_reported",
        "parent consolidated",
        "reconcile with audited/annual reporting",
        "reference_only", "reference_only_no_repo_copy",
        RETRIEVAL_DATE,
        "Management explanations remain company-reported.",
    ),
    (
        "S6BDOC_IDF_010", "Indofood", "PT Indofood Sukses Makmur Tbk",
        "S6A_T08", "period_results", "Full-year financial results FY2023",
        "official_earnings_release", "FY2023", "2024-03-25", "consolidated",
        "https://www.indofood.com/menu/financial-press-releases/indofoods-full-year-financial-results-for-the-year-ended-31-december-2023",
        "",
        "direct_page_verified", "acquired_reference", "2",
        "period-result corroboration", "company_reported",
        "parent consolidated",
        "reconcile with audited/annual reporting",
        "reference_only", "reference_only_no_repo_copy",
        RETRIEVAL_DATE,
        "Management explanations remain company-reported.",
    ),
    (
        "S6BDOC_IDF_011", "Indofood", "PT Indofood Sukses Makmur Tbk",
        "S6A_T08", "period_results", "Full-year financial results FY2024",
        "official_earnings_release", "FY2024", "2025-03-25", "consolidated",
        "https://www.indofood.com/menu/financial-press-releases/indofoods-full-year-financial-results-for-the-year-ended-31-december-2024",
        "",
        "direct_page_verified", "acquired_reference", "2",
        "period-result corroboration", "company_reported",
        "parent consolidated",
        "reconcile with audited/annual reporting",
        "reference_only", "reference_only_no_repo_copy",
        RETRIEVAL_DATE,
        "Management explanations remain company-reported.",
    ),
    (
        "S6BDOC_IDF_012", "Indofood",
        "PT Indofood Sukses Makmur Tbk via First Pacific Company Limited",
        "S6A_T08", "period_results", "Indofood FY2025 results republication",
        "official_parent_regulatory_republication", "FY2025", "2026-03-30",
        "consolidated",
        "https://doc.irasia.com/listco/hk/firstpacific/press/p260330.pdf",
        "https://doc.irasia.com/listco/hk/firstpacific/press/p260330.pdf",
        "official_parent_or_regulatory_republication",
        "verified_reference_only", "3",
        "FY2025 result corroboration", "company_reported",
        "Indofood parent consolidated results republished by listed parent",
        "not a substitute for an Indofood owned-domain release where one becomes available",
        "reference_only", "reference_only_no_repo_copy",
        RETRIEVAL_DATE,
        "Used only to close the FY2025 release-reference gap with an explicit source caveat.",
    ),
    (
        "S6BDOC_IDF_013", "Indofood", "PT Indofood Sukses Makmur Tbk",
        "S6A_T12", "segment_scope", "Bogasari",
        "official_corporate_webpage", "current_page", "", "Indonesia",
        "https://www.indofood.com/business/bogasari", "",
        "official_page_verified", "verified_reference_only", "2",
        "consumer flour and pasta perimeter definition", "company_reported",
        "Bogasari consumer and industrial activities are both described",
        "consumer portfolio interpretation must exclude industrial-only outcomes",
        "reference_only", "reference_only_no_repo_copy",
        RETRIEVAL_DATE,
        "Supports perimeter definition; not a time-series result source.",
    ),
    (
        "S6BDOC_IDF_014", "Indofood", "PT Indofood Sukses Makmur Tbk",
        "S6A_T13", "segment_scope", "Agribusiness",
        "official_corporate_webpage", "current_page", "", "Indonesia",
        "https://www.indofood.com/business/agribusiness", "",
        "official_page_verified", "verified_reference_only", "2",
        "consumer edible-oils/fats perimeter definition", "company_reported",
        "Agribusiness includes upstream and downstream activities",
        "consumer EOF must remain separated from plantations and industrial products",
        "reference_only", "reference_only_no_repo_copy",
        RETRIEVAL_DATE,
        "Supports explicit downstream consumer-versus-upstream separation.",
    ),

    # ------------------------------------------------------------------
    # Indofood CBP
    # ------------------------------------------------------------------
    *[
        (
            f"S6BDOC_ICBP_AR_{year}", "Indofood",
            "PT Indofood CBP Sukses Makmur Tbk",
            "S6A_T09", "annual_reporting",
            f"ICBP Annual Report {year}", "annual_report",
            f"FY{year}", "", "consolidated ICBP",
            "https://www.indofoodcbp.com/investor-relation/annual-report", "",
            "official_index_entry_verified", "verified_reference_only", "1",
            "annual strategy and result source",
            "company_reported",
            "ICBP consolidated; includes disclosed overseas operations",
            "preserve geography and consolidation scope",
            "reference_only", "reference_only_pending_specific_terms",
            RETRIEVAL_DATE,
            "Target-year report is confirmed by official investor materials; direct file URL is not required for Stage 6B.",
        )
        for year in (2022, 2023, 2024, 2025)
    ],
    *[
        (
            f"S6BDOC_ICBP_FS_{year}", "Indofood",
            "PT Indofood CBP Sukses Makmur Tbk",
            "S6A_T10", "audited_financials",
            f"ICBP Annual Financial Statements {year}",
            "audited_financial_statement",
            f"FY{year}", "", "consolidated ICBP",
            "https://www.indofoodcbp.com/menu/financial-statements",
            (
                "https://indofoodcbp.com/uploads/statement/ICBP_billingual_31_dec_25_released.pdf"
                if year == 2025 else ""
            ),
            (
                "direct_document_verified"
                if year == 2025
                else "official_index_entry_verified"
            ),
            "verified_reference_only", "1",
            "audited result source",
            "company_reported_or_audited",
            "ICBP consolidated",
            "preserve overseas and consolidation scope",
            "reference_only", "reference_only_pending_specific_terms",
            RETRIEVAL_DATE,
            (
                "FY2025 direct audited PDF is resolved; earlier years remain official index-level references."
                if year == 2025
                else "Official index confirms the target year; direct file URL remains unresolved."
            ),
        )
        for year in (2022, 2023, 2024, 2025)
    ],
    (
        "S6BDOC_ICBP_009", "Indofood",
        "PT Indofood CBP Sukses Makmur Tbk",
        "S6A_T11", "period_results",
        "ICBP full-year financial results FY2022",
        "official_earnings_release", "FY2022", "2023-03-27", "consolidated ICBP",
        "https://www.indofoodcbp.com/press-release/90_icbps-full-year-financial-results-for-the-year-ended-31-december-2022",
        "",
        "direct_page_verified", "acquired_reference", "2",
        "period-result corroboration", "company_reported",
        "ICBP consolidated",
        "management explanations remain company-reported",
        "reference_only", "reference_only_pending_specific_terms",
        RETRIEVAL_DATE,
        "Reconcile headline results with audited reporting.",
    ),
    (
        "S6BDOC_ICBP_010", "Indofood",
        "PT Indofood CBP Sukses Makmur Tbk",
        "S6A_T11", "period_results",
        "ICBP full-year financial results FY2023",
        "official_earnings_release", "FY2023", "2024-03-25", "consolidated ICBP",
        "https://www.indofoodcbp.com/press-release/97_icbps-full-year-financial-results-for-the-year-ended-31-december-2023",
        "",
        "direct_page_verified", "acquired_reference", "2",
        "period-result corroboration", "company_reported",
        "ICBP consolidated",
        "management explanations remain company-reported",
        "reference_only", "reference_only_pending_specific_terms",
        RETRIEVAL_DATE,
        "Reconcile headline results with audited reporting.",
    ),
    (
        "S6BDOC_ICBP_011", "Indofood",
        "PT Indofood CBP Sukses Makmur Tbk",
        "S6A_T11", "period_results",
        "ICBP full-year financial results FY2024",
        "official_earnings_release", "FY2024", "2025-03-25", "consolidated ICBP",
        "https://www.indofoodcbp.com/menu/financial-press-releases/icbps-full-year-financial-results-for-the-year-ended-31-december-2024",
        "",
        "direct_page_verified", "acquired_reference", "2",
        "period-result and management-explanation source", "company_reported",
        "ICBP consolidated",
        "volume/productivity explanations remain company-reported",
        "reference_only", "reference_only_pending_specific_terms",
        RETRIEVAL_DATE,
        "No causal upgrade is permitted from management attribution.",
    ),
    (
        "S6BDOC_ICBP_012", "Indofood",
        "PT Indofood CBP Sukses Makmur Tbk",
        "S6A_T11", "period_results",
        "ICBP FY2025 annual-report approval confirmation",
        "official_earnings_or_agm_release", "FY2025", "2026-06-26",
        "consolidated ICBP",
        "https://www.indofoodcbp.com/menu/financial-press-releases/icbp-shareholders-approved-all-resolutions-proposed-in-the-agm",
        "",
        "direct_page_verified", "verified_reference_only", "2",
        "FY2025 annual-result availability corroboration", "company_reported",
        "ICBP consolidated",
        "not a substitute for a dedicated full-year earnings release",
        "reference_only", "reference_only_pending_specific_terms",
        RETRIEVAL_DATE,
        "Used together with the FY2025 audited financial statements.",
    ),
    (
        "S6BDOC_ICBP_013", "Indofood",
        "PT Indofood CBP Sukses Makmur Tbk",
        "S6A_T09;S6A_T11", "strategy_events",
        "ICBP Press Releases",
        "official_corporate_press_release_index", "2022–2025 archive", "",
        "Indonesia / disclosed operating scope",
        "https://www.indofoodcbp.com/press-release", "",
        "official_page_verified", "verified_reference_only", "2",
        "dated strategy-event discovery", "company_reported",
        "ICBP and named subsidiaries/brands as stated",
        "article-level scope must be preserved",
        "reference_only", "reference_only_pending_specific_terms",
        RETRIEVAL_DATE,
        "Includes dated product, channel and activation announcements; later extraction must use article-level pages.",
    ),

    # ------------------------------------------------------------------
    # Mayora
    # ------------------------------------------------------------------
    (
        "S6BDOC_MYR_AR_2022", "Mayora", "PT Mayora Indah Tbk",
        "S6A_T14;S6A_T17", "annual_reporting",
        "Annual Report & Sustainability Report 2022", "annual_report",
        "FY2022", "", "Indonesia plus exports",
        "https://www.mayoraindah.co.id/content/laporan-tahunan-mayora-21",
        "https://mayoraindah.co.id/assets/upload/file/arsr-2022.pdf",
        "direct_document_verified", "verified_reference_only", "1",
        "annual strategy, geography and company-result source",
        "company_reported",
        "PT Mayora Indah Tbk and consolidated subsidiaries",
        "preserve domestic/export scope",
        "reference_only", "no_raw_redistribution",
        RETRIEVAL_DATE,
        "Raw PDF must not be mirrored in the repository.",
    ),
    (
        "S6BDOC_MYR_AR_2023", "Mayora", "PT Mayora Indah Tbk",
        "S6A_T14;S6A_T17", "annual_reporting",
        "Annual Report & Sustainability Report 2023", "annual_report",
        "FY2023", "", "Indonesia plus exports",
        "https://www.mayoraindah.co.id/content/laporan-tahunan-mayora-21",
        "https://www.mayoraindah.co.id/assets/upload/file/lt-2023-10e42.pdf",
        "direct_document_verified", "verified_reference_only", "1",
        "annual strategy, geography and company-result source",
        "company_reported",
        "PT Mayora Indah Tbk and consolidated subsidiaries",
        "preserve domestic/export scope",
        "reference_only", "no_raw_redistribution",
        RETRIEVAL_DATE,
        "Raw PDF must not be mirrored in the repository.",
    ),
    (
        "S6BDOC_MYR_AR_2024", "Mayora", "PT Mayora Indah Tbk",
        "S6A_T14;S6A_T17", "annual_reporting",
        "Annual Report & Sustainability Report 2024", "annual_report",
        "FY2024", "", "Indonesia plus exports",
        "https://www.mayoraindah.co.id/content/laporan-tahunan-mayora-21", "",
        "official_index_entry_verified", "verified_reference_only", "1",
        "annual strategy, geography and company-result source",
        "company_reported",
        "PT Mayora Indah Tbk and consolidated subsidiaries",
        "preserve domestic/export scope",
        "reference_only", "no_raw_redistribution",
        RETRIEVAL_DATE,
        "Official index entry verified; direct file URL remains unresolved.",
    ),
    (
        "S6BDOC_MYR_AR_2025", "Mayora", "PT Mayora Indah Tbk",
        "S6A_T14;S6A_T17", "annual_reporting",
        "Annual Report 2025", "annual_report",
        "FY2025", "2026", "Indonesia plus exports",
        "https://www.mayoraindah.co.id/content/laporan-tahunan-mayora-21",
        "https://www.mayoraindah.co.id/assets/upload/file/ar2025-mayora-web-version.pdf",
        "direct_document_verified", "verified_reference_only", "1",
        "annual strategy, geography and company-result source",
        "company_reported",
        "PT Mayora Indah Tbk and consolidated subsidiaries",
        "preserve domestic/export scope",
        "reference_only", "no_raw_redistribution",
        RETRIEVAL_DATE,
        "Official direct document resolved; repository remains reference-only.",
    ),
    *[
        (
            f"S6BDOC_MYR_FS_{year}", "Mayora", "PT Mayora Indah Tbk",
            "S6A_T15;S6A_T17", "audited_financials",
            f"Annual Financial Statements {year}",
            "audited_financial_statement", f"FY{year}", "",
            "consolidated / domestic-export disclosure where reported",
            "https://www.mayoraindah.co.id/id/content/Laporan-Keuangan-Tahunan-23",
            (
                "https://www.mayoraindah.co.id/assets/upload/file/lkk---20241231-a023f.pdf"
                if year == 2024
                else (
                    "https://www.mayoraindah.co.id/assets/upload/file/pt-mayora-indah-tbk-and-its-subsidaries-2025.pdf"
                    if year == 2025
                    else ""
                )
            ),
            (
                "direct_document_verified"
                if year in (2024, 2025)
                else "official_index_entry_verified"
            ),
            "verified_reference_only", "1",
            "audited result and geography source",
            "company_reported_or_audited",
            "PT Mayora Indah Tbk and consolidated subsidiaries",
            "do not interpret mixed geography as Indonesian household demand",
            "reference_only", "no_raw_redistribution",
            RETRIEVAL_DATE,
            (
                "Official direct document resolved; raw redistribution is prohibited."
                if year in (2024, 2025)
                else "Official index entry verified; direct file URL remains unresolved."
            ),
        )
        for year in (2022, 2023, 2024, 2025)
    ],
    (
        "S6BDOC_MYR_009", "Mayora", "PT Mayora Indah Tbk",
        "S6A_T16", "public_disclosures", "Public Expose",
        "public_expose_index", "target-period archive with gaps", "", "Indonesia",
        "https://mayoraindah.co.id/content/Public-Expose-95", "",
        "official_page_verified", "verified_reference_only", "2",
        "strategy and disclosure discovery", "company_reported",
        "PT Mayora Indah Tbk",
        "missing archive years are not interpreted as no action",
        "reference_only", "no_raw_redistribution",
        RETRIEVAL_DATE,
        "Target-period archive coverage is incomplete.",
    ),
    (
        "S6BDOC_MYR_010", "Mayora", "PT Mayora Indah Tbk",
        "S6A_T16", "public_disclosures", "Information Disclosure",
        "regulatory_disclosure_index", "current_archive", "", "transaction-specific",
        "https://mayoraindah.co.id/en/content/Keterbukaan-Informasi-91", "",
        "official_page_verified", "verified_reference_only", "1",
        "ownership/capital-action discovery", "company_reported_or_filing",
        "transaction-specific scope",
        "use filing date and stated transaction perimeter",
        "reference_only", "no_raw_redistribution",
        RETRIEVAL_DATE,
        "Use only dated disclosure records with explicit scope.",
    ),

    # ------------------------------------------------------------------
    # Unilever Indonesia
    # ------------------------------------------------------------------
    *[
        (
            f"S6BDOC_UNV_AR_{year}", "Unilever Indonesia",
            "PT Unilever Indonesia Tbk",
            "S6A_T19;S6A_T21", "annual_reporting",
            f"Annual Report {year}", "annual_report",
            f"FY{year}",
            {
                2022: "2023-05-01",
                2023: "2024-05-01",
                2024: "2025-04-30",
                2025: "2026-04-29",
            }[year],
            "Indonesia",
            "https://www.unilever.co.id/en/investors/annual-financial-and-sustainability-report/annual-reports/",
            {
                2022: "https://www.unilever.co.id/files/d5395e35-a1aa-4965-a059-828b456c10fe/unilever-ar-2022-130723-aqtsgf--1-.pdf",
                2023: "https://www.unilever.co.id/files/indonesia-annual-report-2023.pdf",
                2024: "https://www.unilever.co.id/files/indonesia-annual-reports-2024.pdf",
                2025: "https://www.unilever.co.id/files/annual-reports-2025.pdf",
            }[year],
            "direct_document_verified", "verified_reference_only", "1",
            "annual strategy and company-result source",
            "company_reported",
            "controlled portfolio subject to observation-period ownership",
            "apply time-varying ownership to each extracted result",
            "reference_only",
            "limited_noncommercial_reproduction_no_combination",
            RETRIEVAL_DATE,
            "Direct official PDF resolved; raw report is not stored in the repository.",
        )
        for year in (2022, 2023, 2024, 2025)
    ],
    (
        "S6BDOC_UNV_FS_2023", "Unilever Indonesia",
        "PT Unilever Indonesia Tbk",
        "S6A_T20", "audited_financials",
        "Annual Financial Statements 2023", "audited_financial_statement",
        "FY2023 and FY2022", "2024", "Indonesia",
        "https://www.unilever.co.id/files/unilever-indonesia-annual-financial-statements-q4-2023.pdf",
        "https://www.unilever.co.id/files/unilever-indonesia-annual-financial-statements-q4-2023.pdf",
        "direct_document_verified", "verified_reference_only", "1",
        "audited FY2022–FY2023 accounting source",
        "company_reported_or_audited",
        "PT Unilever Indonesia Tbk",
        "use source-native comparators and accounting definitions",
        "reference_only",
        "limited_noncommercial_reproduction_no_combination",
        RETRIEVAL_DATE,
        "One audited statement supports both FY2022 and FY2023 comparators.",
    ),
    (
        "S6BDOC_UNV_FS_2025", "Unilever Indonesia",
        "PT Unilever Indonesia Tbk",
        "S6A_T20;S6A_T22;S6A_T23", "audited_financials",
        "Annual Financial Statements 2025", "audited_financial_statement",
        "FY2025 and FY2024", "2026", "Indonesia with disclosed export scope",
        "https://www.unilever.co.id/files/indonesia-financial-statements-q4-2025.pdf",
        "https://www.unilever.co.id/files/indonesia-financial-statements-q4-2025.pdf",
        "direct_document_verified", "verified_reference_only", "1",
        "audited FY2024–FY2025 accounting and disposal source",
        "company_reported_or_audited",
        "PT Unilever Indonesia Tbk with discontinued/disposed business treatment",
        "use restated/represented comparators and continuing/disposed scope",
        "reference_only",
        "limited_noncommercial_reproduction_no_combination",
        RETRIEVAL_DATE,
        "Confirms completion of the Ice Cream separation on 8 December 2025.",
    ),
    (
        "S6BDOC_UNV_007", "Unilever Indonesia",
        "PT Unilever Indonesia Tbk",
        "S6A_T21", "results_presentations",
        "H1 2024 Earnings Call Presentation",
        "official_investor_presentation", "H1_2024", "2024", "Indonesia",
        "https://www.unilever.co.id/files/eb672ad8-5e4d-45f0-b4de-b372634ad21c/unvr-earnings-call-h1-2024-final.pdf",
        "https://www.unilever.co.id/files/eb672ad8-5e4d-45f0-b4de-b372634ad21c/unvr-earnings-call-h1-2024-final.pdf",
        "direct_document_verified", "verified_reference_only", "1",
        "strategy and interim-result context", "company_reported",
        "PT Unilever Indonesia Tbk",
        "interim outcomes are not substituted for full-year outcomes",
        "reference_only",
        "limited_noncommercial_reproduction_no_combination",
        RETRIEVAL_DATE,
        "Strategy statements remain company-reported.",
    ),
    (
        "S6BDOC_UNV_008", "Unilever Indonesia",
        "PT Unilever Indonesia Tbk",
        "S6A_T21", "results_presentations",
        "Q1 2025 Earnings Call Presentation",
        "official_investor_presentation", "Q1_2025", "2025", "Indonesia",
        "https://www.unilever.co.id/files/unvr-earnings-call-q1-2025-presentation.pdf",
        "https://www.unilever.co.id/files/unvr-earnings-call-q1-2025-presentation.pdf",
        "direct_document_verified", "verified_reference_only", "1",
        "strategy and interim-result context", "company_reported",
        "PT Unilever Indonesia Tbk",
        "interim outcomes are context-only unless like-for-like rules apply",
        "reference_only",
        "limited_noncommercial_reproduction_no_combination",
        RETRIEVAL_DATE,
        "Product, promotion, place and pricing descriptions remain company-reported.",
    ),
    (
        "S6BDOC_UNV_009", "Unilever Indonesia",
        "PT Unilever Indonesia Tbk",
        "S6A_T22", "ownership_change_context",
        "Completion of SariWangi business sale",
        "material_information_report", "2026_post_window", "2026-03-02", "Indonesia",
        "https://www.unilever.co.id/files/report-on-material-information-or-facts.pdf",
        "https://www.unilever.co.id/files/report-on-material-information-or-facts.pdf",
        "direct_document_verified", "context_only_reference", "1",
        "ownership end-date confirmation", "company_reported_or_filing",
        "SariWangi historical observations valid only while controlled",
        "post-window ownership context only",
        "reference_only",
        "limited_noncommercial_reproduction_no_combination",
        RETRIEVAL_DATE,
        "Confirms completion of the SariWangi sale on 2 March 2026.",
    ),

    # ------------------------------------------------------------------
    # Cross-company conditional context
    # ------------------------------------------------------------------
    (
        "S6BDOC_WPN_001", "Cross-company context",
        "Worldpanel by Numerator",
        "S6A_T25", "consumer_reach",
        "Indonesia Brand Footprint 2024",
        "consumer_panel_report_webpage", "2024_study_edition", "2024-06-28",
        "Indonesia",
        "https://market.worldpanelbynumerator.com/id/News/Indonesia-Brand-Footprint-2024",
        "",
        "inherited_limited_reference", "context_only_reference", "3",
        "conditional consumer-reach context", "external_measurement",
        "research-provider metric only",
        "CRP remains source-native and is not market share",
        "reference_only", "limited_factual_use_only",
        RETRIEVAL_DATE,
        "Inherited limited public reference; no bulk extraction or raw redistribution.",
    ),
]

stage6b_document_registry = pd.DataFrame(
    registry_rows,
    columns=registry_columns,
)

if stage6b_document_registry["document_id"].duplicated().any():
    duplicates = stage6b_document_registry.loc[
        stage6b_document_registry["document_id"].duplicated(keep=False),
        "document_id",
    ].tolist()
    raise RuntimeError(f"Duplicate Stage 6B document IDs: {duplicates}")

if (stage6b_document_registry["source_url"].str.strip() == "").any():
    raise RuntimeError("Every Stage 6B registry row must retain a source URL.")

if set(stage6b_document_registry["repository_storage_policy"]) != {"reference_only"}:
    raise RuntimeError("All Stage 6B source records must remain reference-only.")

valid_target_ids = set(stage6a_universe["target_id"])

for row in stage6b_document_registry.itertuples(index=False):
    mapped_ids = {
        value for value in row.stage6a_target_ids.split(";") if value
    }
    unknown = mapped_ids - valid_target_ids
    if unknown:
        raise RuntimeError(
            f"{row.document_id} maps to unknown Stage 6A targets: {sorted(unknown)}"
        )

print(f"Controlled document references: {len(stage6b_document_registry)}")
print(
    stage6b_document_registry.groupby("canonical_group")
    .size()
    .sort_values(ascending=False)
)


Controlled document references: 56
canonical_group
Indofood                 27
Mayora                   10
Unilever Indonesia        9
Wings Group               9
Cross-company context     1
dtype: int64


## Acquisition-Target Coverage

Map all 26 frozen Stage 6A acquisition targets to the document-level references obtained in Stage 6B. Conditional context targets and disclosure gaps remain explicit rather than being forced into artificial completeness.


In [5]:
coverage_columns = [
    "target_id",
    "canonical_group",
    "evidence_family",
    "target_period",
    "coverage_status",
    "evidence_document_ids",
    "document_count",
    "primary_gap",
    "stage6b_treatment",
]

coverage_spec = {
    "S6A_T01": (
        "covered_with_period_caveat",
        "S6BDOC_WNG_003;S6BDOC_WNG_004;S6BDOC_WNG_005;"
        "S6BDOC_WNG_006;S6BDOC_WNG_007",
        "Official dated action evidence exists for 2022–2023; "
        "2024–2025 archive coverage is not assumed complete.",
        "Proceed to later article-level extraction without interpreting archive gaps as no strategy.",
    ),
    "S6A_T02": (
        "covered_current_context_only",
        "S6BDOC_WNG_001;S6BDOC_WNG_002;S6BDOC_WNG_008;S6BDOC_WNG_009",
        "Current pages cannot establish historical continuity.",
        "Use current portfolio context only and do not backdate.",
    ),
    "S6A_T03": (
        "not_found_with_caveat", "",
        "No Wings annual report was found in official-site discovery.",
        "Retain as disclosure limitation; do not substitute unofficial estimates as equivalent evidence.",
    ),
    "S6A_T04": (
        "not_found_with_caveat", "",
        "No Wings audited financial statements were found in official-site discovery.",
        "Retain as disclosure limitation; missing disclosure is not zero or weak performance.",
    ),
    "S6A_T05": (
        "partial_with_caveat",
        "S6BDOC_WNG_001",
        "Exact operating-entity attribution is not resolved for every action.",
        "Resolve PT Wings Surya / PT Sayap Mas Utama only when a later result attribution requires it.",
    ),
    "S6A_T06": (
        "covered_index_verified",
        "S6BDOC_IDF_AR_2022;S6BDOC_IDF_AR_2023;"
        "S6BDOC_IDF_AR_2024;S6BDOC_IDF_AR_2025",
        "Direct annual-report file URLs are not required at this acquisition stage.",
        "Use official index-confirmed annual reports while preserving parent/segment boundaries.",
    ),
    "S6A_T07": (
        "covered_index_verified",
        "S6BDOC_IDF_FS_2022;S6BDOC_IDF_FS_2023;"
        "S6BDOC_IDF_FS_2024;S6BDOC_IDF_FS_2025",
        "Direct financial-statement file URLs remain unresolved.",
        "Use official index references; obtain document-level facts only in the governed extraction stage.",
    ),
    "S6A_T08": (
        "covered_with_2025_corroboration_caveat",
        "S6BDOC_IDF_009;S6BDOC_IDF_010;S6BDOC_IDF_011;S6BDOC_IDF_012",
        "FY2025 owned-domain full-year release was not resolved; official parent/regulatory republication is used as corroboration.",
        "Keep FY2025 source caveat explicit and reconcile with audited reporting.",
    ),
    "S6A_T09": (
        "covered_index_verified",
        "S6BDOC_ICBP_AR_2022;S6BDOC_ICBP_AR_2023;"
        "S6BDOC_ICBP_AR_2024;S6BDOC_ICBP_AR_2025;"
        "S6BDOC_ICBP_013",
        "Direct annual-report URLs are not required for Stage 6B.",
        "Use official report availability and article-level sources with ICBP perimeter preserved.",
    ),
    "S6A_T10": (
        "covered_with_direct_2025_audited_support",
        "S6BDOC_ICBP_FS_2022;S6BDOC_ICBP_FS_2023;"
        "S6BDOC_ICBP_FS_2024;S6BDOC_ICBP_FS_2025",
        "FY2022–FY2024 direct files remain unresolved while FY2025 is directly resolved.",
        "Preserve consolidated and geography scope.",
    ),
    "S6A_T11": (
        "covered_with_2025_caveat",
        "S6BDOC_ICBP_009;S6BDOC_ICBP_010;S6BDOC_ICBP_011;"
        "S6BDOC_ICBP_012;S6BDOC_ICBP_FS_2025",
        "Dedicated FY2025 full-year earnings release was not resolved.",
        "Use audited FY2025 statements plus AGM confirmation; management explanations require explicit company-reported labels.",
    ),
    "S6A_T12": (
        "covered_scope_verified",
        "S6BDOC_IDF_AR_2022;S6BDOC_IDF_AR_2023;"
        "S6BDOC_IDF_AR_2024;S6BDOC_IDF_AR_2025;S6BDOC_IDF_013",
        "Bogasari mixes consumer and industrial activities.",
        "Extract consumer flour/pasta evidence separately from industrial-only outcomes.",
    ),
    "S6A_T13": (
        "covered_scope_verified_with_upstream_separation",
        "S6BDOC_IDF_AR_2022;S6BDOC_IDF_AR_2023;"
        "S6BDOC_IDF_AR_2024;S6BDOC_IDF_AR_2025;S6BDOC_IDF_014",
        "Agribusiness includes plantations/upstream and downstream consumer EOF.",
        "Extract branded consumer cooking oils/fats separately from upstream and industrial operations.",
    ),
    "S6A_T14": (
        "covered",
        "S6BDOC_MYR_AR_2022;S6BDOC_MYR_AR_2023;"
        "S6BDOC_MYR_AR_2024;S6BDOC_MYR_AR_2025",
        "",
        "Proceed with consolidated-scope extraction while preserving domestic/export distinctions.",
    ),
    "S6A_T15": (
        "covered",
        "S6BDOC_MYR_FS_2022;S6BDOC_MYR_FS_2023;"
        "S6BDOC_MYR_FS_2024;S6BDOC_MYR_FS_2025",
        "Some direct file URLs are unresolved but official index entries are verified.",
        "Use audited definitions and preserve consolidated scope.",
    ),
    "S6A_T16": (
        "covered_with_period_gaps",
        "S6BDOC_MYR_009;S6BDOC_MYR_010",
        "Public-expose archive has target-period gaps.",
        "Use only dated disclosures actually available; absence is not evidence of no strategy.",
    ),
    "S6A_T17": (
        "covered",
        "S6BDOC_MYR_AR_2022;S6BDOC_MYR_AR_2023;"
        "S6BDOC_MYR_AR_2024;S6BDOC_MYR_AR_2025;"
        "S6BDOC_MYR_FS_2022;S6BDOC_MYR_FS_2023;"
        "S6BDOC_MYR_FS_2024;S6BDOC_MYR_FS_2025",
        "",
        "Retain domestic and export measures separately wherever disclosed.",
    ),
    "S6A_T18": (
        "conditional_sensitivity_only", "",
        "PT Tirta Fresindo Jaya / Le Minerale is outside strict-control primary Mayora scope.",
        "Do not acquire or use as primary Mayora evidence; add only if a pre-specified sensitivity analysis requires it.",
    ),
    "S6A_T19": (
        "covered",
        "S6BDOC_UNV_AR_2022;S6BDOC_UNV_AR_2023;"
        "S6BDOC_UNV_AR_2024;S6BDOC_UNV_AR_2025",
        "",
        "Apply observation-period ownership to every later extracted brand/business result.",
    ),
    "S6A_T20": (
        "covered",
        "S6BDOC_UNV_FS_2023;S6BDOC_UNV_FS_2025",
        "",
        "Use source-native restated/represented comparators, geography and discontinued-business treatment.",
    ),
    "S6A_T21": (
        "covered",
        "S6BDOC_UNV_AR_2022;S6BDOC_UNV_AR_2023;"
        "S6BDOC_UNV_AR_2024;S6BDOC_UNV_AR_2025;"
        "S6BDOC_UNV_007;S6BDOC_UNV_008",
        "Interim presentations contain interim outcomes.",
        "Use presentations primarily for documented strategy actions; do not substitute interim outcomes for full-year company results.",
    ),
    "S6A_T22": (
        "covered_context_only",
        "S6BDOC_UNV_FS_2025;S6BDOC_UNV_009",
        "SariWangi completion occurs after the primary FY2022–FY2025 result window.",
        "Use disposal evidence for ownership-period validity only; do not carry disposed businesses into later outcomes.",
    ),
    "S6A_T23": (
        "covered",
        "S6BDOC_UNV_FS_2025",
        "",
        "Use the source-reported comparator and continuing/disposed-business presentation.",
    ),
    "S6A_T24": (
        "conditional_not_acquired", "",
        "Macro/input-cost context is only required when needed to assess a specific disclosed alternative factor.",
        "Defer acquisition until a later extraction/linkage question explicitly requires it.",
    ),
    "S6A_T25": (
        "inherited_limited_context",
        "S6BDOC_WPN_001",
        "Public consumer-reach evidence is limited and copyrighted.",
        "Use only narrow factual context; CRP remains source-native and is not market share.",
    ),
    "S6A_T26": (
        "conditional_not_acquired", "",
        "No market-measurement source is required until a specific later claim needs it.",
        "Acquire only a source that explicitly measures a defined market share; otherwise do not use market-share terminology.",
    ),
}

coverage_rows = []

for target in stage6a_universe.itertuples(index=False):
    if target.target_id not in coverage_spec:
        raise RuntimeError(f"Missing Stage 6B coverage specification: {target.target_id}")

    status, document_ids, gap, treatment = coverage_spec[target.target_id]
    ids = [value for value in document_ids.split(";") if value]

    coverage_rows.append(
        {
            "target_id": target.target_id,
            "canonical_group": target.canonical_group,
            "evidence_family": target.evidence_family,
            "target_period": target.target_period,
            "coverage_status": status,
            "evidence_document_ids": document_ids,
            "document_count": len(ids),
            "primary_gap": gap,
            "stage6b_treatment": treatment,
        }
    )

stage6b_target_coverage = pd.DataFrame(
    coverage_rows,
    columns=coverage_columns,
)

registry_ids = set(stage6b_document_registry["document_id"])

for row in stage6b_target_coverage.itertuples(index=False):
    ids = {value for value in row.evidence_document_ids.split(";") if value}
    unknown = ids - registry_ids
    if unknown:
        raise RuntimeError(
            f"{row.target_id} references unknown Stage 6B documents: {sorted(unknown)}"
        )

if set(stage6b_target_coverage["target_id"]) != valid_target_ids:
    raise RuntimeError("Stage 6B target coverage does not match all 26 Stage 6A targets.")

print(f"Target coverage rows: {len(stage6b_target_coverage)}")
print(stage6b_target_coverage["coverage_status"].value_counts())


Target coverage rows: 26
coverage_status
covered                                            7
covered_index_verified                             3
conditional_not_acquired                           2
not_found_with_caveat                              2
partial_with_caveat                                1
covered_with_2025_corroboration_caveat             1
covered_current_context_only                       1
covered_with_period_caveat                         1
covered_with_2025_caveat                           1
covered_with_direct_2025_audited_support           1
covered_scope_verified_with_upstream_separation    1
covered_scope_verified                             1
covered_with_period_gaps                           1
conditional_sensitivity_only                       1
covered_context_only                               1
inherited_limited_context                          1
Name: count, dtype: int64


## Acquisition Exceptions and Caveat Registry

Record unresolved links, disclosure asymmetries, legal restrictions, period gaps, sensitivity-only scope, and conditional context requirements. Exceptions remain analytical metadata rather than being silently repaired.


In [6]:
exception_columns = [
    "exception_id",
    "affected_target_ids",
    "canonical_group",
    "exception_type",
    "severity",
    "status",
    "description",
    "required_treatment",
]

exception_rows = [
    (
        "S6BEX001", "S6A_T03", "Wings Group",
        "official_disclosure_not_found", "caveat", "open",
        "No Wings annual report was found in official-site discovery for FY2022–FY2025.",
        "Retain disclosure limitation; do not infer zero or weak performance.",
    ),
    (
        "S6BEX002", "S6A_T04", "Wings Group",
        "official_disclosure_not_found", "caveat", "open",
        "No Wings audited financial statements were found in official-site discovery for FY2022–FY2025.",
        "Do not replace audited evidence with unofficial estimates as equivalent evidence.",
    ),
    (
        "S6BEX003", "S6A_T05", "Wings Group",
        "entity_resolution_partial", "caveat", "open",
        "Exact PT Wings Surya / PT Sayap Mas Utama operating-entity attribution is not resolved for every action.",
        "Resolve only when a later result attribution requires legal-entity precision.",
    ),
    (
        "S6BEX004", "S6A_T01", "Wings Group",
        "period_archive_coverage_gap", "caveat", "open",
        "Official dated strategy-event discovery is stronger for 2022–2023 than for 2024–2025.",
        "Do not interpret archive coverage gaps as absence of strategy.",
    ),
    (
        "S6BEX005", "S6A_T06;S6A_T07", "Indofood",
        "direct_file_url_unresolved", "minor", "open",
        "Indofood target-year annual reports and financial statements are verified on official indexes, but direct file URLs are not required/resolved in Stage 6B.",
        "Retain index-level verification and resolve only if later extraction requires file-level access.",
    ),
    (
        "S6BEX006", "S6A_T08", "Indofood",
        "owned_domain_release_gap", "caveat", "open",
        "An Indofood owned-domain FY2025 full-year earnings release was not resolved during Stage 6B discovery.",
        "Use official parent/regulatory republication only as explicit corroboration and reconcile with audited reporting.",
    ),
    (
        "S6BEX007", "S6A_T09;S6A_T10", "Indofood",
        "icbp_direct_file_partial", "minor", "open",
        "ICBP annual-report and earlier financial-statement references are official-index verified; FY2025 audited statements are directly resolved.",
        "Do not treat unresolved direct URLs as missing documents.",
    ),
    (
        "S6BEX008", "S6A_T11", "Indofood",
        "icbp_fy2025_release_gap", "caveat", "open",
        "A dedicated ICBP FY2025 full-year earnings release was not resolved.",
        "Use FY2025 audited statements plus AGM availability confirmation; do not invent management explanations.",
    ),
    (
        "S6BEX009", "S6A_T09;S6A_T10;S6A_T11", "Indofood",
        "reuse_terms_unresolved", "caveat", "open",
        "Specific ICBP content-reuse terms remain unresolved.",
        "Keep all ICBP source documents reference-only and avoid raw republication.",
    ),
    (
        "S6BEX010", "S6A_T14;S6A_T15", "Mayora",
        "direct_file_url_partial", "minor", "open",
        "Some Mayora direct report/financial-statement file URLs remain unresolved despite official index verification.",
        "Treat official index entries as valid acquisition references; resolve only as needed for later extraction.",
    ),
    (
        "S6BEX011", "S6A_T16", "Mayora",
        "period_archive_coverage_gap", "caveat", "open",
        "Mayora public-expose target-period coverage has gaps.",
        "Do not interpret missing public-expose years as absence of company action.",
    ),
    (
        "S6BEX012", "S6A_T14;S6A_T15;S6A_T16;S6A_T17", "Mayora",
        "redistribution_restriction", "caveat", "open",
        "Mayora terms prohibit raw redistribution/systematic downloading without permission.",
        "Preserve reference-only repository treatment and no bulk acquisition.",
    ),
    (
        "S6BEX013", "S6A_T18", "Mayora",
        "sensitivity_only_scope", "caveat", "controlled",
        "PT Tirta Fresindo Jaya / Le Minerale remains outside strict-control primary Mayora scope.",
        "Use only in a separately labelled pre-specified sensitivity analysis.",
    ),
    (
        "S6BEX014", "S6A_T22", "Unilever Indonesia",
        "time_varying_ownership", "caveat", "controlled",
        "Ice Cream separation completed on 8 December 2025; SariWangi sale completed on 2 March 2026.",
        "Apply observation-period ownership and keep SariWangi completion as post-window ownership context.",
    ),
    (
        "S6BEX015", "S6A_T24", "Cross-company context",
        "conditional_context_not_acquired", "informational", "deferred",
        "Macro and input-cost context has not been acquired because no specific later alternative-factor test has yet required it.",
        "Acquire only when a defined linkage question requires this context.",
    ),
    (
        "S6BEX016", "S6A_T25", "Cross-company context",
        "licensed_public_summary_limit", "caveat", "controlled",
        "Consumer-reach context is limited to narrow public factual use.",
        "Do not bulk extract or relabel CRP as market share.",
    ),
    (
        "S6BEX017", "S6A_T26", "Cross-company context",
        "conditional_market_measurement_not_acquired", "informational", "deferred",
        "No market-measurement source has been acquired because no specific Stage 6B requirement calls for it.",
        "Acquire only an explicitly defined market-share source when a later claim requires it.",
    ),
]

stage6b_acquisition_exceptions = pd.DataFrame(
    exception_rows,
    columns=exception_columns,
)

if stage6b_acquisition_exceptions["exception_id"].duplicated().any():
    raise RuntimeError("Duplicate Stage 6B exception IDs detected.")

for row in stage6b_acquisition_exceptions.itertuples(index=False):
    ids = {value for value in row.affected_target_ids.split(";") if value}
    unknown = ids - valid_target_ids
    if unknown:
        raise RuntimeError(
            f"{row.exception_id} references unknown targets: {sorted(unknown)}"
        )

print(f"Acquisition exceptions: {len(stage6b_acquisition_exceptions)}")
print(stage6b_acquisition_exceptions["severity"].value_counts())


Acquisition exceptions: 17
severity
caveat           12
minor             3
informational     2
Name: count, dtype: int64


## Stage 6B Validation

Validate input integrity, frozen target coverage, document provenance, legal/storage boundaries, company/entity scope, ownership-period treatment, missingness, and strict separation between acquisition and later extraction/linkage analysis.


In [7]:
validation_columns = [
    "check_id",
    "validation_area",
    "check_description",
    "result",
    "status",
    "critical_failure",
    "required_treatment",
]

focal_groups = {"Wings Group", "Indofood", "Mayora", "Unilever Indonesia"}
registry_groups = set(stage6b_document_registry["canonical_group"])
coverage_status_counts = stage6b_target_coverage["coverage_status"].value_counts()

validation_rows = [
    (
        "S6B001", "input_integrity",
        "All five canonical Stage 6A inputs match their locked SHA-256 values.",
        "5/5 inputs passed", "passed", "no",
        "Stop Stage 6B if any inherited input differs.",
    ),
    (
        "S6B002", "prior_stage_gate",
        "Stage 6A final gate remains PASS_WITH_CAVEAT.",
        "PASS_WITH_CAVEAT", "passed_with_caveat", "no",
        "Carry Stage 6A caveats into document acquisition.",
    ),
    (
        "S6B003", "target_universe",
        "All 26 frozen Stage 6A acquisition targets are represented exactly once.",
        f"{len(stage6b_target_coverage)}/26 targets", "passed", "no",
        "Do not add or remove targets after observing source availability.",
    ),
    (
        "S6B004", "focal_group_coverage",
        "The controlled registry contains references for all four focal groups.",
        f"{len(focal_groups & registry_groups)}/4 focal groups", "passed", "no",
        "Keep group and legal-entity scope explicit.",
    ),
    (
        "S6B005", "document_identity",
        "Every controlled document/reference has a unique document ID.",
        f"{stage6b_document_registry['document_id'].nunique()} unique IDs",
        "passed", "no",
        "Do not merge distinct documents under one identifier.",
    ),
    (
        "S6B006", "source_url_provenance",
        "Every controlled registry record retains a source URL.",
        "complete", "passed", "no",
        "Never replace authoritative URLs with local raw copies.",
    ),
    (
        "S6B007", "target_mapping",
        "Every document-to-target mapping uses a frozen Stage 6A target ID.",
        "valid", "passed", "no",
        "Reject post-hoc target creation.",
    ),
    (
        "S6B008", "repository_storage",
        "All Stage 6B document records remain reference-only for repository storage.",
        "reference_only", "passed_with_caveat", "no",
        "Do not commit copyrighted source PDFs or mirrored webpages.",
    ),
    (
        "S6B009", "raw_acquisition",
        "Stage 6B creates no raw copyrighted-report directory or repository copy.",
        "none created", "passed", "no",
        "Keep raw source documents outside the repository unless reuse rights are later established.",
    ),
    (
        "S6B010", "bulk_access",
        "Stage 6B performs no bulk/systematic scraping of corporate source sites.",
        "none performed", "passed", "no",
        "Use manual/single-document reference acquisition within source terms.",
    ),
    (
        "S6B011", "wings_strategy_sources",
        "Wings has dated official action references for the primary strategy window.",
        "2022–2023 dated action references acquired",
        "passed_with_caveat", "no",
        "Do not infer complete 2024–2025 coverage from the web archive.",
    ),
    (
        "S6B012", "wings_financial_disclosure",
        "Wings annual report and audited-financial targets retain explicit not-found states.",
        "2 disclosure targets not found",
        "passed_with_caveat", "no",
        "Treat as disclosure asymmetry, not performance evidence.",
    ),
    (
        "S6B013", "wings_entity_scope",
        "Wings operating-entity resolution remains conditional on later attribution need.",
        "partial", "passed_with_caveat", "no",
        "Do not invent legal-entity precision.",
    ),
    (
        "S6B014", "indofood_parent_reporting",
        "Indofood parent annual reports and annual financial statements cover FY2022–FY2025 through official index entries.",
        "FY2022–FY2025 covered", "passed", "no",
        "Preserve consolidated and segment boundaries.",
    ),
    (
        "S6B015", "indofood_period_results",
        "Indofood FY2022–FY2024 owned-domain result releases are acquired and FY2025 has explicit parent/regulatory corroboration.",
        "FY2022–FY2025 referenced", "passed_with_caveat", "no",
        "Keep the FY2025 source caveat explicit.",
    ),
    (
        "S6B016", "indofood_bogasari_scope",
        "Bogasari consumer flour/pasta scope is separated from industrial-only activity.",
        "scope rule retained", "passed", "no",
        "Do not attribute industrial-only outcomes to packaged-FMCG performance.",
    ),
    (
        "S6B017", "indofood_agribusiness_scope",
        "Agribusiness consumer edible oils/fats remain separated from plantations/upstream and industrial products.",
        "scope rule retained", "passed", "no",
        "Extract only consumer-relevant downstream evidence for primary portfolio results.",
    ),
    (
        "S6B018", "icbp_reporting",
        "ICBP annual and audited-financial source families cover FY2022–FY2025.",
        "FY2022–FY2025 covered", "passed_with_caveat", "no",
        "Preserve ICBP consolidation/geography and reference-only legal treatment.",
    ),
    (
        "S6B019", "icbp_claim_attribution",
        "ICBP management explanations remain explicitly company-reported.",
        "retained", "passed", "no",
        "Do not convert management attribution into independent causation.",
    ),
    (
        "S6B020", "mayora_reporting",
        "Mayora annual reports and annual financial statements cover FY2022–FY2025 at official index or direct-document level.",
        "FY2022–FY2025 covered", "passed", "no",
        "Do not confuse unresolved direct URL with unavailable document.",
    ),
    (
        "S6B021", "mayora_geography",
        "Mayora domestic/export separation remains a required later extraction treatment.",
        "retained", "passed", "no",
        "Do not interpret export-inclusive consolidated results as Indonesian household demand.",
    ),
    (
        "S6B022", "mayora_redistribution",
        "Mayora source treatment retains no-raw-redistribution and no-systematic-download restrictions.",
        "retained", "passed_with_caveat", "no",
        "Keep repository reference-only.",
    ),
    (
        "S6B023", "mayora_sensitivity",
        "PT Tirta Fresindo Jaya / Le Minerale remains conditional sensitivity-only.",
        "extended_group_sensitivity_only", "passed", "no",
        "Exclude from strict-control primary Mayora results.",
    ),
    (
        "S6B024", "unilever_annual_reporting",
        "Unilever annual reports are directly resolved for FY2022–FY2025.",
        "4/4 target years direct", "passed", "no",
        "Apply observation-period ownership to later extracted brand/business evidence.",
    ),
    (
        "S6B025", "unilever_financial_reporting",
        "Two direct audited annual financial statements support FY2022–FY2025 accounting coverage.",
        "FY2022–FY2025 covered", "passed", "no",
        "Use source-native comparators, restatements and discontinued-business scope.",
    ),
    (
        "S6B026", "unilever_time_varying_ownership",
        "Ice Cream and SariWangi disposal dates remain explicit ownership-period context.",
        "Ice Cream 2025-12-08; SariWangi 2026-03-02",
        "passed_with_caveat", "no",
        "Do not carry disposed businesses beyond valid ownership periods.",
    ),
    (
        "S6B027", "interim_evidence",
        "Interim presentations are retained as strategy/context evidence rather than substituted for full-year outcomes.",
        "retained", "passed", "no",
        "Apply like-for-like rule before any later interim result comparison.",
    ),
    (
        "S6B028", "consumer_reach_semantics",
        "Worldpanel consumer-reach context remains limited and CRP is not relabelled as market share.",
        "retained", "passed_with_caveat", "no",
        "Keep source-native metric terminology.",
    ),
    (
        "S6B029", "conditional_context",
        "Macro/input-cost and defined market-share sources remain conditional rather than forced into Stage 6B.",
        "2 conditional targets deferred", "passed", "no",
        "Acquire only when a later analytical question requires them.",
    ),
    (
        "S6B030", "missingness",
        "Not-found, unresolved and deferred states are preserved without zero filling or silent substitution.",
        "preserved", "passed", "no",
        "Keep disclosure and access gaps explicit.",
    ),
    (
        "S6B031", "analysis_boundary",
        "Stage 6B performs no analytical observation extraction, strategy-result linkage, ranking, composite scoring or winner selection.",
        "none performed", "passed", "no",
        "Reserve extraction and linkage for governed later stages.",
    ),
    (
        "S6B032", "reporting_boundary",
        "Stage 6B creates no analytical report or README.",
        "none created", "passed", "no",
        "Defer reporting until analysis, validation, visualization and findings are complete.",
    ),
    (
        "S6B033", "stage_gate",
        "Controlled official-source acquisition is complete enough to begin governed evidence extraction.",
        "PASS_WITH_CAVEAT", "passed_with_caveat", "no",
        "Carry Wings disclosure asymmetry, unresolved direct-link gaps, legal restrictions and ownership-period caveats into Stage 6C.",
    ),
]

stage6b_acquisition_validation = pd.DataFrame(
    validation_rows,
    columns=validation_columns,
)

expected_checks = {f"S6B{i:03d}" for i in range(1, 34)}

if set(stage6b_acquisition_validation["check_id"]) != expected_checks:
    raise RuntimeError("Stage 6B validation registry is incomplete.")

if (stage6b_acquisition_validation["critical_failure"] == "yes").any():
    raise RuntimeError("Stage 6B contains a critical validation failure.")

final_stage6b = stage6b_acquisition_validation.loc[
    stage6b_acquisition_validation["check_id"] == "S6B033"
].iloc[0]

if (
    final_stage6b["result"] != "PASS_WITH_CAVEAT"
    or final_stage6b["status"] != "passed_with_caveat"
):
    raise RuntimeError("Unexpected Stage 6B final gate.")

print(f"Validation checks: {len(stage6b_acquisition_validation)}")
print(
    "Critical failures: "
    f"{(stage6b_acquisition_validation['critical_failure'] == 'yes').sum()}"
)
print(
    f"Final gate: {final_stage6b['result']} / {final_stage6b['status']}"
)


Validation checks: 33
Critical failures: 0
Final gate: PASS_WITH_CAVEAT / passed_with_caveat


## Canonical Stage 6B Outputs

Write the five canonical Stage 6B metadata outputs. No source document binaries, raw webpages, reports, README, or analytical report are written.


In [8]:
output_frames = {
    "metadata/stage6b_input_lock.csv": stage6b_input_lock,
    "metadata/stage6b_document_registry.csv": stage6b_document_registry,
    "metadata/stage6b_target_coverage.csv": stage6b_target_coverage,
    "metadata/stage6b_acquisition_exceptions.csv": stage6b_acquisition_exceptions,
    "metadata/stage6b_acquisition_validation.csv": stage6b_acquisition_validation,
}

for relative_path, dataframe in output_frames.items():
    destination = OUTPUT_ROOT / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    dataframe.to_csv(destination, index=False, encoding="utf-8")

print("Canonical Stage 6B outputs written:")
for relative_path, dataframe in output_frames.items():
    print(f"  {relative_path}: {len(dataframe)} rows")


Canonical Stage 6B outputs written:
  metadata/stage6b_input_lock.csv: 5 rows
  metadata/stage6b_document_registry.csv: 56 rows
  metadata/stage6b_target_coverage.csv: 26 rows
  metadata/stage6b_acquisition_exceptions.csv: 17 rows
  metadata/stage6b_acquisition_validation.csv: 33 rows


## Output Manifest

Create a deterministic SHA-256 manifest for the five canonical Stage 6B metadata outputs. The manifest excludes itself.


In [9]:
manifest_rows = []

for relative_path, dataframe in output_frames.items():
    path = OUTPUT_ROOT / relative_path
    manifest_rows.append(
        {
            "file_path": relative_path,
            "artifact_type": "csv",
            "row_count": len(dataframe),
            "sha256": sha256_file(path),
            "locked_input_commit": INPUT_COMMIT,
        }
    )

stage6b_output_manifest = pd.DataFrame(manifest_rows)

manifest_path = OUTPUT_ROOT / "metadata/stage6b_output_manifest.csv"
stage6b_output_manifest.to_csv(
    manifest_path,
    index=False,
    encoding="utf-8",
)

print(f"Manifest-tracked outputs: {len(stage6b_output_manifest)}")
display(stage6b_output_manifest)


Manifest-tracked outputs: 5


,file_path,artifact_type,row_count,sha256,locked_input_commit
0,metadata/stage6b_input_lock.csv,csv,5,87575313eab43fec7b4f289495fc002a42a3e7ab5b36ef089f65e810e8f095e4,51e5b3d9459469aa34c91adcc2fd2d4573a0170c
1,metadata/stage6b_document_registry.csv,csv,56,3b872109b5f721a438b9c62bf7c400b658db25900fee5b7607e5684f72abcebd,51e5b3d9459469aa34c91adcc2fd2d4573a0170c
2,metadata/stage6b_target_coverage.csv,csv,26,19e315ef8d36899740e27cb65a2c85cdff6ad2c3a4a6f4147d74633387904a3b,51e5b3d9459469aa34c91adcc2fd2d4573a0170c
3,metadata/stage6b_acquisition_exceptions.csv,csv,17,580fbb95336df70fde63ae4ac9f7123b130f0cf0ea3e946a71ce47c4354f28a1,51e5b3d9459469aa34c91adcc2fd2d4573a0170c
4,metadata/stage6b_acquisition_validation.csv,csv,33,3aba60ef0fcfcadef169cd542919488c873a735dd9f946bdfe93f15f63279308,51e5b3d9459469aa34c91adcc2fd2d4573a0170c


## Final Stage 6B Quality Assurance

Re-read the generated files, verify row counts and manifest hashes, confirm that no raw source documents were created, and print the final acquisition gate.


In [10]:
expected_row_counts = {
    "metadata/stage6b_input_lock.csv": len(stage6b_input_lock),
    "metadata/stage6b_document_registry.csv": len(stage6b_document_registry),
    "metadata/stage6b_target_coverage.csv": 26,
    "metadata/stage6b_acquisition_exceptions.csv": len(stage6b_acquisition_exceptions),
    "metadata/stage6b_acquisition_validation.csv": 33,
}

for relative_path, expected_rows in expected_row_counts.items():
    reloaded = pd.read_csv(
        OUTPUT_ROOT / relative_path,
        dtype=str,
        keep_default_na=False,
    )
    if len(reloaded) != expected_rows:
        raise RuntimeError(
            f"{relative_path}: expected {expected_rows} rows, found {len(reloaded)}."
        )

manifest_reloaded = pd.read_csv(
    OUTPUT_ROOT / "metadata/stage6b_output_manifest.csv",
    dtype=str,
    keep_default_na=False,
)

if len(manifest_reloaded) != 5:
    raise RuntimeError(
        f"Expected 5 Stage 6B manifest rows, found {len(manifest_reloaded)}."
    )

hash_failures = []

for row in manifest_reloaded.itertuples(index=False):
    actual_hash = sha256_file(OUTPUT_ROOT / row.file_path)
    if actual_hash != row.sha256:
        hash_failures.append(row.file_path)

if hash_failures:
    raise RuntimeError(
        f"Stage 6B output-manifest hash mismatch: {hash_failures}"
    )

all_runtime_files = [
    path for path in OUTPUT_ROOT.rglob("*") if path.is_file()
]

unexpected_binary_extensions = {
    ".pdf", ".doc", ".docx", ".xls", ".xlsx", ".ppt", ".pptx",
    ".jpg", ".jpeg", ".png", ".webp", ".zip",
}

unexpected_binaries = [
    str(path)
    for path in all_runtime_files
    if path.suffix.lower() in unexpected_binary_extensions
]

if unexpected_binaries:
    raise RuntimeError(
        "Unexpected raw/binary source files were created: "
        f"{unexpected_binaries}"
    )

if set(stage6b_target_coverage["target_id"]) != valid_target_ids:
    raise RuntimeError("Final target coverage changed unexpectedly.")

if stage6b_document_registry["document_id"].duplicated().any():
    raise RuntimeError("Duplicate document IDs found during final QA.")

print("Stage 6B final QA passed.")
print(f"Locked Stage 6A inputs: {len(stage6b_input_lock)}")
print(f"Controlled document references: {len(stage6b_document_registry)}")
print(f"Frozen targets covered/retained: {len(stage6b_target_coverage)}/26")
print(f"Acquisition exceptions/caveats: {len(stage6b_acquisition_exceptions)}")
print(f"Validation checks: {len(stage6b_acquisition_validation)}")
print("Critical failures: 0")
print("Raw copyrighted source files written: 0")
print("Manifest hashes: 5/5 matched")
print(
    f"Stage 6B gate: "
    f"{final_stage6b['result']} / {final_stage6b['status']}"
)

display(
    stage6b_target_coverage[
        [
            "target_id",
            "canonical_group",
            "coverage_status",
            "document_count",
            "primary_gap",
        ]
    ]
)


Stage 6B final QA passed.
Locked Stage 6A inputs: 5
Controlled document references: 56
Frozen targets covered/retained: 26/26
Acquisition exceptions/caveats: 17
Validation checks: 33
Critical failures: 0
Raw copyrighted source files written: 0
Manifest hashes: 5/5 matched
Stage 6B gate: PASS_WITH_CAVEAT / passed_with_caveat


,target_id,canonical_group,coverage_status,document_count,primary_gap
0,S6A_T01,Wings Group,covered_with_period_caveat,5,Official dated action evidence exists for 2022–2023; 2024–2025 archive coverage is not assumed complete.
1,S6A_T02,Wings Group,covered_current_context_only,4,Current pages cannot establish historical continuity.
2,S6A_T03,Wings Group,not_found_with_caveat,0,No Wings annual report was found in official-site discovery.
3,S6A_T04,Wings Group,not_found_with_caveat,0,No Wings audited financial statements were found in official-site discovery.
4,S6A_T05,Wings Group,partial_with_caveat,1,Exact operating-entity attribution is not resolved for every action.
5,S6A_T06,Indofood,covered_index_verified,4,Direct annual-report file URLs are not required at this acquisition stage.
6,S6A_T07,Indofood,covered_index_verified,4,Direct financial-statement file URLs remain unresolved.
7,S6A_T08,Indofood,covered_with_2025_corroboration_caveat,4,FY2025 owned-domain full-year release was not resolved; official parent/regulatory republication is used as corrobor...
8,S6A_T09,Indofood,covered_index_verified,5,Direct annual-report URLs are not required for Stage 6B.
9,S6A_T10,Indofood,covered_with_direct_2025_audited_support,4,FY2022–FY2024 direct files remain unresolved while FY2025 is directly resolved.
